# MOONATLAS · NASA-IBM LFM inference (science smoke test)

Runs the **released NASA-IBM Lunar Foundation Model checkpoints** on one held-out SomBench test sample per task
(WAC crater detection, polar ice prospectivity, IMP segmentation), then normalizes, validates and exports the results.

This notebook only orchestrates. All logic lives in the `moonatlas_science` package and runs the same outside Colab.

**Integrity rules:** if any inference step fails, the run stops. Reference labels are never substituted for model output.

**Runtime:** `Runtime → Change runtime type → T4 GPU` (any NVIDIA GPU works; CPU also works, slowly).

## STEP 0 · Check the runtime

In [ ]:
import subprocess
import sys

print(sys.version)
assert sys.version_info[:2] in ((3, 11), (3, 12)), "NASA-IBM upstream requires Python 3.11 or 3.12"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No GPU detected: CPU run (slow)")

## STEP 1 · Clone MOONATLAS Science

Clones the public repository; no token is needed.


In [ ]:
REPO = "https://github.com/RRG1312/moonatlas-science.git"
BRANCH = "main"

import os

if not os.path.exists("/content/moonatlas-science"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, "/content/moonatlas-science"], check=True)
os.chdir("/content/moonatlas-science")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


## STEP 2 · Install dependencies

Checks out the NASA-IBM upstream code at the pinned commit and installs it with its requirements
(PyTorch ≥ 2.12, TerraTorch, Lightning, rasterio…). Takes ~3–6 minutes. Later steps run as separate
processes, so no runtime restart is needed.

In [ ]:
!pip install -q -e ".[dev]"
!python scripts/setup_environment.py

## STEP 3 · Download the NASA-IBM checkpoints (~6.3 GB)

Backbone 2.4 GB (required to construct every model) + crater 1.1 GB + IMP 1.1 GB + ice 1.7 GB, pinned to exact
Hugging Face revisions and verified by sha256. Takes ~3–8 minutes.

In [ ]:
!python -m moonatlas_science.steps.download_models

## STEP 4 · Download the SomBench smoke-test samples (~20 MB)

In [ ]:
!python -m moonatlas_science.steps.download_data

## STEP 5 · Run the smoke inference

One held-out test sample per task, through the official TerraTorch configs. Each takes well under a minute on a GPU
(model construction dominates). Raw outputs go to `data/artifacts/raw/`.

In [ ]:
!python -m moonatlas_science.steps.run_crater_inference
!python -m moonatlas_science.steps.run_ice_inference
!python -m moonatlas_science.steps.run_imp_inference

## STEP 6 · Production inference

**Not enabled in this phase.** Batch inference over the held-out test splits starts only after the smoke-test outputs
are reviewed. This cell intentionally does nothing.

In [ ]:
print("Production batch inference is pending review of the smoke test.")

## STEP 7 · Build normalized MOONATLAS outputs, validate and review

In [ ]:
!python -m moonatlas_science.steps.process_craters
!python -m moonatlas_science.steps.process_ice
!python -m moonatlas_science.steps.process_imp
!python -m moonatlas_science.steps.build_ice_mosaic
!python -m moonatlas_science.steps.build_catalog
!python -m moonatlas_science.steps.build_manifest --scope smoke
!python -m moonatlas_science.steps.validate_data
!python -m moonatlas_science.steps.sanity_report
!python -m pytest -q

In [ ]:
from pathlib import Path

from IPython.display import Image, display

for figure in sorted(Path("data/artifacts/sanity").glob("*.png")):
    print(figure.name)
    display(Image(filename=str(figure), width=1100))

## STEP 8 · Export the generated files

Creates `moonatlas-science-smoke.zip` with `data/build/` (normalized outputs), `data/artifacts/raw/` (raw model
outputs) and `data/artifacts/sanity/` (figures + report), and downloads it. Optionally copies it to Google Drive.

In [ ]:
import shutil
import zipfile
from pathlib import Path

archive = Path("/content/moonatlas-science-smoke.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for root in ("data/build", "data/artifacts/raw", "data/artifacts/sanity"):
        for path in Path(root).rglob("*"):
            if path.is_file():
                zf.write(path, path.as_posix())
print(f"{archive} · {archive.stat().st_size / 1e6:.1f} MB")

COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy(archive, "/content/drive/MyDrive/moonatlas-science-smoke.zip")

from google.colab import files

files.download(str(archive))